## Formatting for Plots

In [3]:
import sys
from pathlib import Path
root = Path().resolve()
src_path = root / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from analysislib import formatting
formatting.format_notebook()

---

## Part 3 — Shape, Size, and Structure

---

<div style="width: 1200px">

In Part 2, we discovered that galaxies split into two distinct colour populations — a blue, star-forming cloud and a red, quenched sequence — with surprisingly few galaxies in between. Colour alone tells us a great deal about a galaxy's evolutionary state. But it raises an immediate question:

**Does a galaxy's *shape* tell the same story as its colour?**

Intuitively, we might expect that blue star-forming galaxies look different from red quenched ones. And indeed they do — spiral galaxies, with their sweeping disc structures and active star-forming arms, tend to be blue; elliptical galaxies, smooth and featureless, tend to be red. But the relationship is not perfect, and the exceptions are often the most scientifically interesting cases.

In this notebook, we move beyond colour and begin measuring **structure** — how spread out the light is, how centrally concentrated the galaxy is, and how these properties relate to evolutionary state.

</div>

---

### The Petrosian Radius — "How Spread Out is the Light?"

---

<div style="width: 1200px">

Measuring the size of a galaxy is harder than it sounds. Unlike a ball with a clear edge, galaxies fade gradually into the background — there is no sharp boundary where a galaxy ends and empty space begins. We need a robust, distance-independent way to define "how big" a galaxy is.

The solution the SDSS uses is called the **Petrosian radius**, and the idea is elegant. Rather than asking "where does the galaxy end?", we ask a different question:

> *At what radius does the galaxy's surface brightness drop to a fixed fraction of the average brightness inside that radius?*

Concretely, the Petrosian radius $r_P$ is defined as the radius at which the local surface brightness is 20% of the mean surface brightness enclosed within that radius. Think of it as the point where the galaxy starts to "fade into the background" in a well-defined, reproducible way.

Why does this matter? Because the Petrosian radius is **independent of distance** — a galaxy at $z = 0.05$ and the same galaxy at $z = 0.15$ will give the same Petrosian radius in physical units (kiloparsecs), even though the more distant one appears smaller on the sky. This makes it ideal for comparing galaxy sizes across our sample.

In our dataset, this is stored as `petroRad_r` — the Petrosian radius measured in the $r$ band, in arcseconds.

Two other size measurements also appear in the data:

* **`petroR50`** — the radius enclosing 50% of the total Petrosian flux (the "half-light radius")
* **`petroR90`** — the radius enclosing 90% of the total Petrosian flux

The ratio between these two is the key to our next measurement.

</div>

---

### The Concentration Index — "Is the Light Piled Up in the Centre?"

---

<div style="width: 1200px">

Knowing how big a galaxy is tells us one thing. Knowing *where* the light is distributed tells us something much richer about its structure.

The **concentration index** $C$ captures this:

$$C = 5 \times \log_{10}\left(\frac{r_{90}}{r_{50}}\right)$$

where $r_{90}$ is the radius containing 90% of the light and $r_{50}$ is the radius containing 50% of the light. A galaxy where the light is tightly packed into the centre has a large $r_{90}/r_{50}$ ratio — most of the light is already captured within $r_{50}$, so you need to go much further out to get the remaining 40%. A galaxy where the light is spread smoothly across the disc has a smaller ratio.

In practice:

* **$C \gtrsim 3.0$** — highly concentrated, bulge-dominated: **elliptical galaxies**
* **$C \approx 2.3$–$3.0$** — intermediate concentration: **lenticular (S0) galaxies**  
* **$C \lesssim 2.3$** — diffuse, disc-dominated: **spiral galaxies**

This single number gives us a morphological classification without ever looking at a galaxy image.

</div>

---

### The Sérsic Index — "What Shape is the Light Profile?"

---

<div style="width: 1200px">

A complementary way to describe a galaxy's light distribution is the **Sérsic index** $n$, which describes the *shape* of the brightness profile — how quickly the light fades as you move outward from the centre.

The Sérsic profile is:

$$I(r) \propto \exp\left[-b_n\left(\left(\frac{r}{r_e}\right)^{1/n} - 1\right)\right]$$

where $r_e$ is the half-light radius. The index $n$ controls the curvature:

* **$n = 1$** — an exponential profile, characteristic of **spiral galaxy discs**: the light fades smoothly and steadily outward
* **$n = 4$** — the de Vaucouleurs profile, characteristic of **elliptical galaxies**: a very bright, compact core with extended wings that fade slowly
* **$n$ between 1 and 4** — a mix of disc and bulge, typical of **lenticular and intermediate-type galaxies**

We won't fit Sérsic profiles directly in this notebook, but the SDSS provides two related profile fits in the data: the exponential fit (`expRad_r`) and the de Vaucouleurs fit (`deVRad_r`). The column `fracDeV_r` tells us what fraction of the light is better described by the de Vaucouleurs profile — essentially a proxy for the Sérsic index. A value of `1.0` means a pure elliptical-like profile; `0.0` means a pure disc.

Now let's see all of this in the data.

</div>

---

### Setting Up

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import gaussian_kde
from scipy.signal import find_peaks
from IPython.display import display, HTML

display(HTML("""
<style>
  .widget-label { color: #cccccc !important; font-family: monospace !important; }
  .jupyter-widgets { background: transparent !important; }
</style>
"""))

# Load and clean
df = pd.read_csv("data/SDSS_500k_v5.csv")
df = df[df["zWarning"] == 0]
for band in ["u", "g", "r", "i", "z"]:
    df = df[df[band] > 0]

galaxies = df[df["class"] == "GALAXY"].copy()
galaxies = galaxies[
    (galaxies["petroRad_r"] > 0) &
    (galaxies["fracDeV_r"]  >= 0)
].copy()

# Derived columns
galaxies["g_r"] = galaxies["g"] - galaxies["r"]
galaxies["C"]   = 5 * np.log10(galaxies["petroR90_r"] / galaxies["petroR50_r"])

# Trim to physically sensible ranges
galaxies = galaxies[
    (galaxies["g_r"] > -0.2) & (galaxies["g_r"] < 1.4) &
    (galaxies["C"]   >  1.5) & (galaxies["C"]   < 4.5) &
    (galaxies["petroRad_r"] < 200)
].copy()

# z < 0.05 subset for colour-reliable plots
gal_lowz = galaxies[galaxies["redshift"] < 0.05].copy()

# Colour classes based on g-r
# Using peak positions from Part 2: blue cloud < 0.58, red sequence > 0.76
gal_lowz["colour_class"] = pd.cut(
    gal_lowz["g_r"],
    bins=[-np.inf, 0.58, 0.76, np.inf],
    labels=["Blue Cloud", "Green Valley", "Red Sequence"]
)

blue = gal_lowz[gal_lowz["colour_class"] == "Blue Cloud"]
green = gal_lowz[gal_lowz["colour_class"] == "Green Valley"]
red  = gal_lowz[gal_lowz["colour_class"] == "Red Sequence"]

print(f"Full galaxy sample:      {len(galaxies):,}")
print(f"Low-z sample (z < 0.05): {len(gal_lowz):,}")
print(f"  Blue Cloud:   {len(blue):,}")
print(f"  Green Valley: {len(green):,}")
print(f"  Red Sequence: {len(red):,}")

KeyError: 'petroR90_r'

---

### Plot 1 — Concentration Index by Colour Class

---

<div style="width: 1200px">

If galaxy colour and morphology are genuinely linked — as we expect from decades of visual classification studies — then blue and red galaxies should have systematically different concentration indices. Let's check.

</div>

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5), facecolor="#1e1e1e")
ax.set_facecolor("#1e1e1e")

x_range = np.linspace(1.5, 4.5, 800)

populations = [
    (blue,  "#74c7ec", "Blue Cloud"),
    (green, "#69db7c", "Green Valley"),
    (red,   "#f38ba8", "Red Sequence"),
]

for pop, color, label in populations:
    c_vals = pop["C"].dropna().values
    if len(c_vals) < 50:
        continue
    kde = gaussian_kde(c_vals, bw_method=0.08)
    kde_vals = kde(x_range)
    kde_vals = kde_vals / kde_vals.max()
    ax.fill_between(x_range, 0, kde_vals, color=color, alpha=0.18, zorder=2)
    ax.plot(x_range, kde_vals, lw=2, color=color, label=label, zorder=3)
    # Mark the median
    med = np.median(c_vals)
    med_h = kde([med])[0] / kde(x_range).max()
    ax.vlines(med, 0, med_h, color=color, lw=1.2, linestyle="--", alpha=0.6)
    ax.text(med, med_h + 0.04, f"median\n{med:.2f}",
            color=color, fontsize=7.5, ha="center", va="bottom")

# Morphology reference lines
for x_ref, lbl in [(2.3, "← spirals"), (3.0, "ellipticals →")]:
    ax.axvline(x_ref, color="#555555", lw=1, linestyle=":")
    ax.text(x_ref + 0.04, 1.12, lbl, color="#666666",
            fontsize=7.5, ha="left", style="italic")

ax.set_xlim(1.5, 4.5)
ax.set_ylim(0, 1.3)
ax.set_yticks([])
ax.set_xlabel(r"Concentration index  $C = 5\,\log_{10}(r_{90}/r_{50})$",
              color="#aaaaaa", fontsize=10)
ax.set_ylabel("Relative count", color="#aaaaaa", fontsize=10)
ax.set_title("Concentration Index by Colour Class — morphology encoded in a single number",
             color="#e0e0e0", fontsize=11, loc="left", pad=12)
ax.tick_params(colors="#777777", labelsize=8)
ax.grid(True, linestyle=":", alpha=0.12, color="#ffffff")
ax.legend(facecolor="#2a2a2a", edgecolor="#444444",
          labelcolor="#e0e0e0", fontsize=9)
for spine in ax.spines.values(): spine.set_edgecolor("#333333")

plt.tight_layout()
plt.savefig("figures/concentration_by_colour.png", dpi=150,
            bbox_inches="tight", facecolor="#1e1e1e")
plt.show()

<div style="width: 1200px">

The separation is clear and physically interpretable:

* **Blue Cloud galaxies** peak at lower $C$ — their light is spread across an extended disc, consistent with spiral morphology. The concentration distribution is broad, reflecting the variety of spiral types from loose, open-armed galaxies to tighter, bulge-heavy ones.

* **Red Sequence galaxies** peak at significantly higher $C$ — their light is tightly concentrated toward the centre, the hallmark of elliptical galaxies that have used up or expelled their gas and settled into a smooth, compact structure.

* **Green Valley galaxies** sit in between, with a broad distribution spanning both morphological types. This is consistent with the idea that these are transitioning galaxies — some may have already developed a dominant bulge while still retaining some disc structure; others may be discs whose star formation is being quenched without a major structural transformation.

The dashed reference lines at $C = 2.3$ and $C = 3.0$ mark the conventional boundaries between spiral, lenticular, and elliptical morphologies.

</div>

---

### Plot 2 — The Four Quadrants: Concentration vs Colour

---

<div style="width: 1200px">

The concentration distribution tells us that colour and morphology are correlated — but how tight is that relationship, and what do the outliers look like?

Plotting concentration against $g - r$ colour directly creates a 2D map with four naturally defined quadrants, each telling a different physical story:

| Quadrant | Colour | Concentration | Interpretation |
|----------|--------|---------------|----------------|
| Bottom-left | Blue | Low | Classic **spiral galaxies** — disc-dominated, actively star-forming |
| Top-right | Red | High | Classic **elliptical galaxies** — bulge-dominated, quenched |
| Top-left | Blue | High | **Compact star-forming galaxies** — rare, possibly post-merger starbursts |
| Bottom-right | Red | Low | **Passive disc galaxies** (S0/lenticulars) — quenched but disc structure retained |

The top-left and bottom-right quadrants are the most scientifically interesting: they represent galaxies where colour and morphology have decoupled, hinting at different quenching pathways.

</div>

In [ ]:
sample = gal_lowz.sample(min(30000, len(gal_lowz)), random_state=42)

x = sample["g_r"].values
y = sample["C"].values

xy = np.vstack([x, y])
density = gaussian_kde(xy)(xy)
sort_idx = density.argsort()
x, y, density = x[sort_idx], y[sort_idx], density[sort_idx]
log_density = np.log1p(density)
log_density = (log_density - log_density.min()) / (log_density.max() - log_density.min())

# Quadrant boundaries
c_split  = 2.75   # concentration boundary
gr_split = 0.65   # colour boundary (green valley centre)

fig, ax = plt.subplots(figsize=(10, 7), facecolor="#1e1e1e")
ax.set_facecolor("#1e1e1e")

sc = ax.scatter(x, y, c=log_density, s=2, alpha=0.7,
                cmap="inferno", rasterized=True)

# Quadrant dividers
ax.axvline(gr_split, color="#69db7c", lw=1.2, linestyle="--", alpha=0.5)
ax.axhline(c_split,  color="#888888", lw=1.2, linestyle="--", alpha=0.5)

label_props = dict(
    fontsize=9, ha="center", va="center",
    bbox=dict(boxstyle="round,pad=0.4", fc="#1e1e1e", ec="#444444", alpha=0.75)
)

# Quadrant labels
ax.text(0.32, 1.9,  "Spiral galaxies\n(disc, star-forming)",
        color="#74c7ec", **label_props)
ax.text(0.95, 3.6,  "Elliptical galaxies\n(bulge, quenched)",
        color="#f38ba8", **label_props)
ax.text(0.32, 3.6,  "Compact star-formers\n(post-merger starbursts?)",
        color="#cba6f7", **label_props)
ax.text(0.95, 1.9,  "Passive discs\n(lenticular / S0)",
        color="#fab387", **label_props)

# Green valley band
ax.axvspan(0.58, 0.76, color="#69db7c", alpha=0.05)
ax.text(0.67, 4.35, "Green Valley", color="#69db7c",
        fontsize=7.5, ha="center", style="italic")

cbar = plt.colorbar(sc, ax=ax, pad=0.01)
cbar.set_label("Log point density", color="#aaaaaa", fontsize=8)
cbar.ax.yaxis.set_tick_params(color="#777777")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="#777777", fontsize=7)
cbar.outline.set_edgecolor("#333333")

ax.set_xlabel("g − r colour index", color="#aaaaaa", fontsize=10)
ax.set_ylabel(r"Concentration index $C$", color="#aaaaaa", fontsize=10)
ax.set_title("Concentration vs Colour — four quadrants, four evolutionary stories",
             color="#e0e0e0", fontsize=11, loc="left", pad=12)
ax.set_xlim(-0.1, 1.35)
ax.set_ylim(1.5, 4.5)
ax.tick_params(colors="#777777", labelsize=8)
ax.grid(True, linestyle=":", alpha=0.10, color="#ffffff")
for spine in ax.spines.values(): spine.set_edgecolor("#333333")
ax.text(1.33, 1.55, "z < 0.05 sample", color="#555555",
        fontsize=7, ha="right", style="italic")

plt.tight_layout()
plt.savefig("figures/concentration_vs_colour.png", dpi=150,
            bbox_inches="tight", facecolor="#1e1e1e")
plt.show()

<div style="width: 1200px">

The density map reveals the two main populations as bright concentrations in the bottom-left (blue, diffuse spirals) and top-right (red, compact ellipticals) quadrants. The diagonal emptiness between them corresponds to the green valley — galaxies don't linger in the intermediate colour-and-concentration space.

The two off-diagonal quadrants are sparsely populated but scientifically rich:

* **Top-left (blue + compact):** These are rare galaxies — probably compact star-forming systems, possibly triggered by mergers that simultaneously concentrated the stellar mass and ignited a burst of star formation. Some may be early-type galaxies that recently accreted cold gas.

* **Bottom-right (red + diffuse):** These are **passive disc galaxies**, sometimes called S0 or lenticular galaxies. They have retained their disc structure — perhaps through a gentle, internal quenching process — but have stopped forming stars. Their existence suggests that quenching does not always require a dramatic structural transformation.

This last point is important: **you can quench a galaxy without destroying its disc**. The existence of the bottom-right quadrant is one of the key observational clues that quenching mechanisms must include gentle, internal processes — not just violent mergers.

</div>

---

### Plot 3 — Petrosian Radius vs Redshift

---

<div style="width: 1200px">

So far we've been comparing galaxies by their *apparent* sizes — how big they look in the sky, measured in arcseconds. But apparent size depends on distance: a nearby dwarf galaxy can look just as large as a distant giant. To compare the true physical sizes of galaxies, we need to account for this.

Plotting the Petrosian radius against redshift does two things at once:

1. It shows us the **survey geometry** — how the observable population changes with distance (the flux limit biting in at higher $z$)
2. It reveals a real **physical size trend** — do galaxies at different redshifts have different physical extents?

We colour the points by $g - r$ to see whether red and blue galaxies have systematically different sizes.

</div>

In [ ]:
sample_z = galaxies[
    (galaxies["redshift"] > 0.005) &
    (galaxies["redshift"] < 0.30)  &
    (galaxies["petroRad_r"] < 60)
].sample(min(60000, len(galaxies)), random_state=42)

fig, ax = plt.subplots(figsize=(10, 5.5), facecolor="#1e1e1e")
ax.set_facecolor("#1e1e1e")

# Colour points by g-r, clipped to visible range
gr_clipped = sample_z["g_r"].clip(-0.1, 1.3)
sc = ax.scatter(
    sample_z["redshift"], sample_z["petroRad_r"],
    c=gr_clipped, cmap="coolwarm_r",
    vmin=0.1, vmax=1.2,
    s=1.2, alpha=0.4, rasterized=True
)

# Overlay median trend lines per colour class
z_bins = np.linspace(0.01, 0.30, 20)
z_mids = 0.5 * (z_bins[:-1] + z_bins[1:])

for subset, color, label in [
    (galaxies[(galaxies["redshift"] < 0.30) & (galaxies["g_r"] < 0.58)],
     "#74c7ec", "Blue Cloud median"),
    (galaxies[(galaxies["redshift"] < 0.30) & (galaxies["g_r"] > 0.76)],
     "#f38ba8", "Red Sequence median"),
]:
    medians = []
    for lo, hi in zip(z_bins[:-1], z_bins[1:]):
        bin_data = subset[
            (subset["redshift"] >= lo) &
            (subset["redshift"] <  hi) &
            (subset["petroRad_r"] < 60)
        ]["petroRad_r"]
        medians.append(np.median(bin_data) if len(bin_data) > 10 else np.nan)
    ax.plot(z_mids, medians, color=color, lw=2, label=label, zorder=5)

cbar = plt.colorbar(sc, ax=ax, pad=0.01)
cbar.set_label("g − r colour", color="#aaaaaa", fontsize=8)
cbar.ax.yaxis.set_tick_params(color="#777777")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="#777777", fontsize=7)
cbar.outline.set_edgecolor("#333333")

ax.set_xlabel("Redshift (z)", color="#aaaaaa", fontsize=10)
ax.set_ylabel("Petrosian radius (arcsec)", color="#aaaaaa", fontsize=10)
ax.set_title("Petrosian Radius vs Redshift — survey geometry and physical size trends",
             color="#e0e0e0", fontsize=11, loc="left", pad=12)
ax.set_xlim(0, 0.30)
ax.set_ylim(0, 60)
ax.tick_params(colors="#777777", labelsize=8)
ax.grid(True, linestyle=":", alpha=0.10, color="#ffffff")
ax.legend(facecolor="#2a2a2a", edgecolor="#444444",
          labelcolor="#e0e0e0", fontsize=8, loc="upper right")
for spine in ax.spines.values(): spine.set_edgecolor("#333333")

plt.tight_layout()
plt.savefig("figures/petrorad_vs_redshift.png", dpi=150,
            bbox_inches="tight", facecolor="#1e1e1e")
plt.show()

<div style="width: 1200px">

Three features of this plot are worth unpacking:

**The upper envelope drops with redshift.** At low $z$, we see galaxies spanning a wide range of apparent sizes — from tiny compact objects to large extended ones. At higher $z$, the largest apparent sizes disappear. This is partly geometric (physically large galaxies subtend smaller angles at larger distances) and partly the flux limit biting in (only the most luminous, often smaller, galaxies are detectable at high $z$).

**The lower envelope stays roughly flat.** The smallest apparent sizes we detect stay roughly constant with redshift. This reflects the seeing limit of the telescope — galaxies smaller than roughly 1–2 arcseconds cannot be well-resolved and are excluded by the Petrosian algorithm.

**Red galaxies are systematically smaller in apparent size than blue ones** at the same redshift (visible in the median trend lines). This is a real physical effect: elliptical galaxies tend to be more physically compact than spiral galaxies of comparable luminosity. Red sequence galaxies have concentrated their stellar mass into smaller volumes — consistent with the higher concentration indices we saw in Plot 1.

</div>

---

### Bonus Plot — The de Vaucouleurs Fraction: Disc vs Bulge

---

<div style="width: 1200px">

The column `fracDeV_r` gives us something complementary to the concentration index: a direct measure of how bulge-like vs disc-like the galaxy's light profile is, as determined by fitting two standard profile shapes to each object.

A value of `1.0` means the light is best described by a de Vaucouleurs ($n=4$) profile — compact, centrally concentrated, elliptical-like. A value of `0.0` means the light follows an exponential ($n=1$) disc profile. Values in between represent mixed systems.

Critically, `fracDeV_r` is determined from the *shape* of the light profile rather than just its spatial extent, making it an independent morphological measurement from $C$. Plotting the two against each other — coloured by $g-r$ — tests whether they agree.

</div>

In [ ]:
sample_fdev = gal_lowz[
    (gal_lowz["fracDeV_r"] >= 0) &
    (gal_lowz["fracDeV_r"] <= 1)
].sample(min(25000, len(gal_lowz)), random_state=42)

x = sample_fdev["fracDeV_r"].values
y = sample_fdev["C"].values
c = sample_fdev["g_r"].clip(0.0, 1.2).values

# Sort by g-r so blue points don't hide under red
sort_idx = np.argsort(c)
x, y, c = x[sort_idx], y[sort_idx], c[sort_idx]

fig, ax = plt.subplots(figsize=(10, 6), facecolor="#1e1e1e")
ax.set_facecolor("#1e1e1e")

sc = ax.scatter(x, y, c=c, cmap="coolwarm_r", vmin=0.1, vmax=1.1,
                s=2, alpha=0.5, rasterized=True)

# Annotate the corners
corner_props = dict(
    fontsize=8.5, bbox=dict(boxstyle="round,pad=0.35",
                            fc="#1e1e1e", ec="#444444", alpha=0.8)
)
ax.text(0.05, 4.2, "Disc profile\n+ compact",
        color="#cba6f7", ha="left", **corner_props)
ax.text(0.92, 4.2, "Bulge profile\n+ compact\n(ellipticals)",
        color="#f38ba8", ha="center", **corner_props)
ax.text(0.05, 1.65, "Disc profile\n+ diffuse\n(spirals)",
        color="#74c7ec", ha="left", **corner_props)
ax.text(0.92, 1.65, "Bulge profile\n+ diffuse",
        color="#fab387", ha="center", **corner_props)

ax.axvline(0.5, color="#555555", lw=1, linestyle=":")
ax.axhline(2.75, color="#555555", lw=1, linestyle=":")

cbar = plt.colorbar(sc, ax=ax, pad=0.01)
cbar.set_label("g − r colour (blue → red)", color="#aaaaaa", fontsize=8)
cbar.ax.yaxis.set_tick_params(color="#777777")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="#777777", fontsize=7)
cbar.outline.set_edgecolor("#333333")

ax.set_xlabel("de Vaucouleurs fraction  (0 = pure disc,  1 = pure bulge)",
              color="#aaaaaa", fontsize=10)
ax.set_ylabel(r"Concentration index $C$", color="#aaaaaa", fontsize=10)
ax.set_title("Profile Shape vs Concentration — two independent morphology measures agree",
             color="#e0e0e0", fontsize=11, loc="left", pad=12)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(1.5, 4.5)
ax.tick_params(colors="#777777", labelsize=8)
ax.grid(True, linestyle=":", alpha=0.10, color="#ffffff")
for spine in ax.spines.values(): spine.set_edgecolor("#333333")
ax.text(1.03, 1.55, "z < 0.05 sample",
        color="#555555", fontsize=7, ha="right", style="italic")

plt.tight_layout()
plt.savefig("figures/fracdev_vs_concentration.png", dpi=150,
            bbox_inches="tight", facecolor="#1e1e1e")
plt.show()

<div style="width: 1200px">

The two morphological measurements agree remarkably well. Galaxies with high `fracDeV_r` (bulge-like profiles) also tend to have high $C$ (centrally concentrated light), and they are coloured red. Galaxies with low `fracDeV_r` (disc-like profiles) tend to have low $C$ and are coloured blue.

This agreement is reassuring — it tells us that the morphological signal we're measuring is robust. The `fracDeV_r` parameter is derived from profile *shape* fitting, while $C$ is derived purely from the radii enclosing fixed light fractions. They are computed in completely different ways, yet they point to the same physical picture.

Notice also that `fracDeV_r` shows a **bimodal distribution** — galaxies tend to cluster near `0` or near `1`, with fewer in the intermediate range. This mirrors the colour bimodality we saw in Part 2, and reinforces the idea that galaxy morphology, like colour, is not a smooth continuum but a bimodal distribution driven by distinct evolutionary pathways.

</div>

---

### What We've Learned

---

<div style="width: 1200px">

Structure and colour tell a consistent story — but with important nuance.

The main findings from this notebook:

* **Colour and morphology are correlated but not identical.** Blue galaxies tend to be disc-dominated spirals; red galaxies tend to be bulge-dominated ellipticals. But the off-diagonal quadrants — passive discs and compact star-formers — tell us that quenching and structural transformation are not always the same process.

* **The concentration index $C$ is a powerful single-number morphological classifier.** It cleanly separates the blue and red populations in our sample, with green valley galaxies spanning both, consistent with a structural transformation happening alongside (or after) quenching.

* **The Petrosian radius reveals survey geometry.** The apparent size vs redshift plot is not just a data quality diagnostic — it encodes real physics. Red galaxies are systematically more compact than blue ones at the same redshift, a direct consequence of their different formation histories.

* **The de Vaucouleurs fraction independently confirms the morphological picture.** Two different ways of measuring structure — profile shape and concentration — agree with each other and with colour. The physical signal is real.

In the next notebook, we move from structure to **distance** — using redshift and luminosity to place our galaxies in space and time, and beginning to trace how galaxy properties have evolved across cosmic history.

</div>